# Weighted operators

Usually, you should not need to care about this class of operators. *Weighted operators* go through the abstraction that Kafi Streams has scaffolded around DBSP/pydbsp for developer experience. But they can still be useful for certain use cases, or just for toying around and learning about DBSP.

Whereas the stateless and stateful operators explained above do not let the central concept behind DBSP, *ZSets*, shine through, the weighted operators (currently all stateless, by the way) let you access the *weights* of the records going through a Kafi Streams topology.

The "low-level" aspect of these operators is reflected in their `_` prefix. For instance, `map()` is the classic map operator, whereas `_map()` is the weighted version of it etc.

In the examples, we'll change the way how we push inputs and receive outputs: We explicitly associate a weight (`1`) with each input record, and let Kafi Streams also associate its returned records with their respective weights.


## Overview

* [_map()](#_map-operator)
* [_peek()](#_peek-operator)
* [_neg()](#_neg-operator)
* [_flatmap()](#_flatmap-operator)
* [_filter()](#_filter-operator)


## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

<a id="_map-operator"></a>
## _map()

The `_map()` operator is the weighted version of the `map()` operator:

```
_map(_map_fun, **kwargs)
```
* `_map_fun: r, w -> tuple(r, w)` the map function; gets an input record and its weight, does some processing and returns an pair of the output record and its weight.

Here is an example.

In [ ]:
# <------------------------------>
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    ._map(lambda r, w: ({"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]}, -w))
).from_zSet(Tn._to_records)
# <------------------------------>

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


As you can see from the topology, contrary to the `map_fun` of the `map()` operator, the `_map_fun` of the `_map()` operator has two parameters: the input record and its weight.

The `_map_fun` also has two instead of one return value: the output record and the output weight.

What we do in this example is to negate the weight of the record. Hence, in the output, the records all have weight `-1` instead of `1`.


<a id="_peek-operator"></a>
## _peek()

The `_peek()` operator is the weighted version of the `peek()` operator:

```
_peek(prefix_str=None, _peek_fun=None, **kwargs)
```
* `prefix_str` prefix of the printed output (if `_peek_fun` is `None`)
* `_peek_fun: r, w -> None` the `None`-returning peek function getting the input record and its weight

Here are a few examples. The first sets none of the parameters:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._peek()
    # <------------------------------>
)

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]

_ = built_tn.process({click_source_str: input_m_w_tuple_list})


You can see that `_peek` just printed out each of the records coming in plus its weight (always `1`).

Next, we use `_peek` with `prefix_str` set:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._peek("_peek")
    # <------------------------------>
)

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]

_ = built_tn.process({click_source_str: input_m_w_tuple_list})


...and you can see that `_peek()` now adds the prefix `_peek: ` to the input records and their weights printed out.

Last example: We use the `_peek_fun`:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._peek(_peek_fun=lambda r, w: print(f"{r, w}\n"))
    # <------------------------------>
)

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]

_ = built_tn.process({click_source_str: input_m_w_tuple_list})



What we did in the `_peek_fun` is to add a newline after each printed out input record and weight.

<a id="_neg-operator"></a>
## _neg()

This is syntactic sugar for a `_map()` where the input record stays unchanged and its weight is negated.

```
_neg(**kwargs):
```

Here is the example.


In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._neg()
    # <------------------------------>
).from_zSet(Tn._to_records)

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


<a id="_flatmap-operator"></a>
## _flatmap()

The `_flatmap()` operator is the weighted version of the `flatmap()` operator:

```
_flatmap(_flatmap_fun, **kwargs)
```
* `_flatmap_fun: r, w -> set(tuple(r, w))` the flatmap function; gets an input record and its weight and returns a set of pairs of output records and their weight.

Examples:

In [ ]:
built_tn = Tn.build(
    Tn.source(customer_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._flatmap(lambda r, w: {(name_part_str, -w) for name_part_str in r["value"]["name"].split(" ")})
    # <------------------------------>
).from_zSet(Tn._to_records)

input_m_w_tuple_list = [(m, 1) for m in customer_generator.generate(5)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

output_m_w_tuple_list = built_tn.process({customer_source_str: input_m_w_tuple_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


In this example, from each customer input record, we take the `name` field of its `value`, split it and return the set of the parts of the name. So e.g., `"Alexis Jones"` becomes `{"Alexis", "Jones"}`.

In addition to the example for `flatmap()`, in this `_flatmap()` example, we also negate the weight of the input record to `-1`.


<a id="_filter-operator"></a>
## _filter()

The `_filter()` operator is the weighted version of the `filter()` operator:

```
_filter(_filter_fun, **kwargs)
```
* `_filter_fun: r, w -> bool` the filter function; gets an input record and its weight and returns `True` if the record shall be kept or `False` if it shall be discarded.

Examples.


In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._filter(lambda r, w: r["value"]["view_time"] > 60 and (str(w) in str(r["value"]["view_time"])))
    # <------------------------------>
).from_zSet(Tn._to_records)

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(5)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)



In the example, we just keep those records where `view_time` is greater than `60`.

We also pose the rather artificial constraint to only keep those records whose weight (here: always `1`) is a digit that is contained in the digits of the `view_time` ;-)

The idea behind this example is just to show that in the lambda functions of Kafi Streams, you can virtually do everything (ok, you should use only pure functions because only that keeps the topology consistent).

In any case, quite a bit more powerful than pure SQL, no? ;-)